# 猪行为检测 · Colab 基线训练（全流程版）

数据集在 Colab 上直接下载（国内网络无法访问谷歌存储，不要在本地尝试）。

使用步骤：
1. 菜单 **代码执行程序 → 更改运行时类型 → T4 GPU**
2. 从上到下依次运行所有单元格（约 1.5–3 小时）
3. 最后一个单元格自动下载 `results.zip`，拿回本地

> ⚠️ 本笔记本含你的 Roboflow API key，分享或上传 git 前请先删除。

In [ ]:
# 确认 GPU 可用（应看到 Tesla T4）
!nvidia-smi

In [ ]:
# 从 Roboflow 下载数据集（导出链接 15 分钟过期，所以每次现取）
API_KEY = 'YOUR_ROBOFLOW_API_KEY'  # 你的 Roboflow Private API key

import json, glob, zipfile, urllib.request, subprocess

meta = json.load(urllib.request.urlopen(
    f'https://api.roboflow.com/km-sd0ce/pig-behavior-wlvku/1/yolov8?api_key={API_KEY}'))
print('数据集:', meta['project']['name'], '| 图片:', meta['version']['images'],
      '| 切分:', meta['version']['splits'])

subprocess.run(['curl', '-sL', '-o', '/content/dataset.zip', meta['export']['link']], check=True)
with zipfile.ZipFile('/content/dataset.zip') as z:
    z.extractall('/content/dataset')

DATA_YAML = glob.glob('/content/dataset/**/data.yaml', recursive=True)[0]
print('data.yaml 位于:', DATA_YAML)

In [ ]:
# 标注质检：随机抽 12 张训练图画出标注框（结果存进 results.zip，拿回本地复查）
import random, glob, cv2
import matplotlib.pyplot as plt
from pathlib import Path

imgs = random.sample(glob.glob('/content/dataset/**/train/images/*.*', recursive=True), 12)
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, p in zip(axes.flat, imgs):
    img = cv2.imread(p)
    h, w = img.shape[:2]
    lb = Path(p.replace('images', 'labels')).with_suffix('.txt')
    if lb.exists():
        for line in lb.read_text().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            _, xc, yc, bw, bh = map(float, parts[:5])
            x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
            x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(Path(p).name[:20], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig('/content/qc_samples.png', dpi=80)
plt.show()
print('若大量框错位/类别可疑，先不要训练，把 qc_samples.png 发回检查')

In [ ]:
!pip install -q ultralytics
import ultralytics
print('ultralytics', ultralytics.__version__)

In [ ]:
# 基线训练：YOLOv11n，100 轮，640 输入（T4 约 1–2 小时）
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
model.train(data=DATA_YAML, epochs=100, imgsz=640, batch=16,
            device=0, project='/content/results', name='baseline')

In [ ]:
# 验证集评估 + 保存指标
import json

metrics = model.val()
summary = {
    'mAP50': round(float(metrics.box.map50), 4),
    'mAP50-95': round(float(metrics.box.map), 4),
    'precision': round(float(metrics.box.mp), 4),
    'recall': round(float(metrics.box.mr), 4),
}
with open('/content/results/baseline/metrics.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(summary)

In [ ]:
# 打包结果（权重/曲线/指标/质检图）并下载
import shutil
from google.colab import files

shutil.copy('/content/qc_samples.png', '/content/results/baseline/qc_samples.png')
shutil.make_archive('/content/results', 'zip', '/content/results')
files.download('/content/results.zip')